In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/src/ner') if Path('/kaggle/working/src/ner').is_dir() else Path.cwd()
DATASET_DIR = PROJECT_ROOT / 'data' / 'few_nerd_mountains_output'
SPLITS = ['train', 'validation', 'test']

assert DATASET_DIR.exists(), f'Not found: {DATASET_DIR}'
print(DATASET_DIR)
print(sorted(p.name for p in DATASET_DIR.iterdir()))


In [ ]:
info = json.loads((DATASET_DIR / 'train' / 'dataset_info.json').read_text(encoding='utf-8'))
print('features:', list(info['features'].keys()))
print('split sizes:', {k: v['num_examples'] for k, v in info['splits'].items()})
print('BIO mapping assumption: 0=O, 1=B-Mountain, 2=I-Mountain')

In [ ]:
def load_arrow_rows(split_name: str):
    arrow_path = DATASET_DIR / split_name / 'data-00000-of-00001.arrow'
    with pa.memory_map(str(arrow_path), 'r') as source:
        reader = ipc.open_stream(source)
        table = reader.read_all()
    return table.to_pylist()


def extract_mountain_names(tokens, labels):
    names = []
    current = []

    for token, label in zip(tokens, labels):
        if label == 1:
            if current:
                names.append(' '.join(current))
            current = [token]
        elif label == 2:
            if current:
                current.append(token)
        else:
            if current:
                names.append(' '.join(current))
                current = []

    if current:
        names.append(' '.join(current))

    return names


def normalize_name(name: str) -> str:
    return ' '.join(name.lower().strip().split())

In [ ]:
rows_by_split = {split: load_arrow_rows(split) for split in SPLITS}
print({split: len(rows) for split, rows in rows_by_split.items()})

In [ ]:
name_counters = {}
unique_names = {}
sentence_counts = {}

for split, rows in rows_by_split.items():
    counter = Counter()
    mountain_sentences = 0

    for row in rows:
        names = extract_mountain_names(row['tokens'], row['labels'])
        if names:
            mountain_sentences += 1
        for name in names:
            counter[normalize_name(name)] += 1

    name_counters[split] = counter
    unique_names[split] = set(counter)
    sentence_counts[split] = mountain_sentences

pd.DataFrame([
    {
        'split': split,
        'rows_total': len(rows_by_split[split]),
        'rows_with_mountain': sentence_counts[split],
        'unique_mountain_names': len(unique_names[split]),
        'total_mountain_mentions': sum(name_counters[split].values()),
    }
    for split in SPLITS
])

In [ ]:
overlap_rows = []

for left in SPLITS:
    for right in SPLITS:
        inter = unique_names[left] & unique_names[right]
        overlap_rows.append({
            'left': left,
            'right': right,
            'shared_unique_names': len(inter),
            'left_coverage_pct': round(100 * len(inter) / len(unique_names[left]), 2) if unique_names[left] else 0.0,
            'right_coverage_pct': round(100 * len(inter) / len(unique_names[right]), 2) if unique_names[right] else 0.0,
        })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df

In [ ]:
pivot_shared = overlap_df.pivot(index='left', columns='right', values='shared_unique_names')
pivot_shared

In [ ]:
train_val = sorted(unique_names['train'] & unique_names['validation'])
train_test = sorted(unique_names['train'] & unique_names['test'])
val_test = sorted(unique_names['validation'] & unique_names['test'])
all_three = sorted(unique_names['train'] & unique_names['validation'] & unique_names['test'])

summary = pd.DataFrame([
    {'pair': 'train vs validation', 'shared_unique_names': len(train_val), 'sample_names': ', '.join(train_val[:20])},
    {'pair': 'train vs test', 'shared_unique_names': len(train_test), 'sample_names': ', '.join(train_test[:20])},
    {'pair': 'validation vs test', 'shared_unique_names': len(val_test), 'sample_names': ', '.join(val_test[:20])},
    {'pair': 'all three', 'shared_unique_names': len(all_three), 'sample_names': ', '.join(all_three[:20])},
])
summary

In [ ]:
name_presence = []
all_names = sorted(unique_names['train'] | unique_names['validation'] | unique_names['test'])

for name in all_names:
    name_presence.append({
        'name': name,
        'train_mentions': name_counters['train'].get(name, 0),
        'validation_mentions': name_counters['validation'].get(name, 0),
        'test_mentions': name_counters['test'].get(name, 0),
        'in_train': name in unique_names['train'],
        'in_validation': name in unique_names['validation'],
        'in_test': name in unique_names['test'],
    })

presence_df = pd.DataFrame(name_presence)
presence_df.head(30)

In [ ]:
only_train = sorted(unique_names['train'] - unique_names['validation'] - unique_names['test'])
only_validation = sorted(unique_names['validation'] - unique_names['train'] - unique_names['test'])
only_test = sorted(unique_names['test'] - unique_names['train'] - unique_names['validation'])

exclusive_df = pd.DataFrame([
    {'split': 'train_only', 'count': len(only_train), 'sample_names': ', '.join(only_train[:20])},
    {'split': 'validation_only', 'count': len(only_validation), 'sample_names': ', '.join(only_validation[:20])},
    {'split': 'test_only', 'count': len(only_test), 'sample_names': ', '.join(only_test[:20])},
])
exclusive_df